## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import our modules
from src.data_sampler import StratifiedSampler
from src.text_chunker import TextChunker
from src.embedder import EmbeddingGenerator
from src.vector_store import VectorStoreManager

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("All modules imported successfully!")

## 1. Load Filtered Data

In [ ]:
# Load the filtered data from Task 1
data_path = Path('../data/processed/filtered_complaints.csv')

if not data_path.exists():
    raise FileNotFoundError(
        "Filtered data not found. Please run 01_eda_preprocessing.ipynb first."
    )

df = pd.read_csv(data_path)
print(f"Loaded {len(df):,} filtered complaints")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Create Stratified Sample

We'll create a sample of 10,000-15,000 complaints while maintaining product distribution.

In [ ]:
# Initialize sampler
sampler = StratifiedSampler(df, product_column='Product')

# Create sample of 12,000 complaints
SAMPLE_SIZE = 12000
RANDOM_STATE = 42

sample_df = sampler.create_stratified_sample(
    sample_size=SAMPLE_SIZE,
    random_state=RANDOM_STATE
)

print(f"Created sample with {len(sample_df):,} complaints")

In [ ]:
# Validate sample distribution
validation_report = sampler.validate_sample_distribution(
    sample_df,
    tolerance=0.05
)

print("\nSample Distribution Validation:")
print(f"Within tolerance: {validation_report['within_tolerance']}")
print(f"Max deviation: {validation_report['max_deviation']:.4f}")
print("\nDistribution comparison:")
print(validation_report['distribution_comparison'])

In [ ]:
# Visualize distribution comparison
dist_comp = validation_report['distribution_comparison']

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(dist_comp))
width = 0.35

ax.bar(x - width/2, dist_comp['original_proportion'], width, label='Original', alpha=0.8)
ax.bar(x + width/2, dist_comp['sample_proportion'], width, label='Sample', alpha=0.8)

ax.set_xlabel('Product Category')
ax.set_ylabel('Proportion')
ax.set_title('Product Distribution: Original vs Sample')
ax.set_xticks(x)
ax.set_xticklabels(dist_comp.index, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Sample maintains product distribution proportions")

## 3. Text Chunking

Split complaint narratives into smaller chunks for better embedding quality.

In [ ]:
# Initialize text chunker with chosen parameters
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

chunker = TextChunker(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

print(f"Chunker initialized:")
print(f"  - Chunk size: {CHUNK_SIZE} characters")
print(f"  - Overlap: {CHUNK_OVERLAP} characters")
print(f"\nRationale: 500 chars ≈ 100-150 words balances context and granularity.")
print(f"Overlap preserves context at chunk boundaries.")

In [ ]:
# Chunk the sample documents
metadata_columns = [
    'Product',
    'Sub-product',
    'Issue',
    'Company',
    'State',
    'Date received'
]

# Only keep columns that exist in the dataframe
available_metadata = [col for col in metadata_columns if col in sample_df.columns]

chunks_df = chunker.chunk_documents(
    sample_df,
    text_column='Consumer complaint narrative',
    metadata_columns=available_metadata
)

print(f"\nCreated {len(chunks_df):,} chunks from {len(sample_df):,} documents")
print(f"Average chunks per document: {len(chunks_df)/len(sample_df):.2f}")

In [ ]:
# Get and display chunking statistics
chunk_stats = chunker.get_chunk_statistics(chunks_df)

print("\nChunk Statistics:")
for key, value in chunk_stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value:,}")

In [ ]:
# Visualize chunk length distribution
chunk_lengths = chunks_df['text'].str.len()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1.hist(chunk_lengths, bins=50, edgecolor='black', alpha=0.7)
ax1.axvline(chunk_lengths.mean(), color='red', linestyle='--', label=f'Mean: {chunk_lengths.mean():.0f}')
ax1.axvline(CHUNK_SIZE, color='green', linestyle='--', label=f'Max: {CHUNK_SIZE}')
ax1.set_xlabel('Chunk Length (characters)')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Chunk Lengths')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Box plot
ax2.boxplot(chunk_lengths, vert=True)
ax2.set_ylabel('Chunk Length (characters)')
ax2.set_title('Chunk Length Box Plot')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Show example chunks from a single document
example_idx = 0
example_chunks = chunks_df[chunks_df['original_index'] == example_idx]

print(f"\nExample: Document {example_idx} split into {len(example_chunks)} chunks")
print(f"Product: {example_chunks.iloc[0]['Product']}")
print("\nChunk 0:")
print(example_chunks.iloc[0]['text'])
if len(example_chunks) > 1:
    print("\nChunk 1:")
    print(example_chunks.iloc[1]['text'])

## 4. Generate Embeddings

Use sentence-transformers to create vector embeddings for each chunk.

In [ ]:
# Initialize embedding generator
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {MODEL_NAME}")
print("This may take a minute on first run...")

embedder = EmbeddingGenerator(model_name=MODEL_NAME)

print(f"\nModel loaded successfully!")
print(f"Embedding dimension: {embedder.embedding_dim}")
print(f"\nModel choice rationale:")
print(f"  - all-MiniLM-L6-v2 is fast and efficient (80MB)")
print(f"  - 384 dimensions balance quality and size")
print(f"  - Trained on 1B+ sentence pairs")

In [ ]:
# Generate embeddings for all chunks
print(f"Generating embeddings for {len(chunks_df):,} chunks...")
print("This will take several minutes...\n")

embeddings = embedder.batch_embed(
    chunks_df['text'].tolist(),
    batch_size=32,
    show_progress_bar=True
)

print(f"\n✓ Generated embeddings with shape: {embeddings.shape}")
print(f"  {embeddings.shape[0]:,} chunks × {embeddings.shape[1]} dimensions")

In [ ]:
# Visualize embedding space (first 2 dimensions via PCA)
from sklearn.decomposition import PCA

# Sample for visualization
sample_size = min(1000, len(embeddings))
sample_indices = np.random.choice(len(embeddings), sample_size, replace=False)
sample_embeddings = embeddings[sample_indices]
sample_products = chunks_df.iloc[sample_indices]['Product'].values

# Reduce to 2D
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(sample_embeddings)

# Plot
plt.figure(figsize=(12, 8))
products = chunks_df['Product'].unique()
colors = plt.cm.Set3(np.linspace(0, 1, len(products)))

for i, product in enumerate(products):
    mask = sample_products == product
    plt.scatter(
        embeddings_2d[mask, 0],
        embeddings_2d[mask, 1],
        c=[colors[i]],
        label=product,
        alpha=0.6,
        s=20
    )

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('Embedding Space Visualization (PCA)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nNote: Similar complaints should cluster together in embedding space.")

## 5. Build and Persist Vector Store

Store embeddings in ChromaDB for efficient similarity search.

In [ ]:
# Initialize vector store manager
VECTOR_STORE_DIR = "../vector_store"
COLLECTION_NAME = "complaint_chunks"

print(f"Initializing vector store...")
print(f"  Directory: {VECTOR_STORE_DIR}")
print(f"  Collection: {COLLECTION_NAME}")

vector_store = VectorStoreManager(
    persist_directory=VECTOR_STORE_DIR,
    collection_name=COLLECTION_NAME,
    reset=True  # Start fresh
)

print("\n✓ Vector store initialized")

In [ ]:
# Add documents to vector store in batches
print(f"Adding {len(chunks_df):,} chunks to vector store...")

vector_store.add_documents_batch(
    chunks_df=chunks_df,
    embeddings=embeddings,
    text_column='text',
    metadata_columns=available_metadata + ['chunk_index', 'total_chunks', 'original_index'],
    batch_size=1000
)

print("\n✓ All documents added to vector store")

In [ ]:
# Verify vector store
stats = vector_store.get_collection_stats()

print("\nVector Store Statistics:")
print(f"  Collection: {stats['name']}")
print(f"  Total documents: {stats['count']:,}")
print(f"  Persist directory: {stats['persist_directory']}")

## 6. Test Similarity Search

Verify the vector store works by running test queries.

In [ ]:
# Test query
test_query = "I was charged unexpected fees on my credit card"

print(f"Test Query: '{test_query}'\n")

# Generate query embedding
query_embedding = embedder.generate_embedding(test_query)

# Search
results = vector_store.search(
    query_embedding=query_embedding,
    top_k=5
)

print(f"Found {len(results['documents'])} similar chunks:\n")

for i, (doc, metadata, distance) in enumerate(
    zip(results['documents'], results['metadatas'], results['distances'])
):
    print(f"Result {i+1} (distance: {distance:.4f})")
    print(f"Product: {metadata.get('Product', 'N/A')}")
    print(f"Text: {doc[:200]}...")
    print("-" * 80)
    print()

In [ ]:
# Test filtered search
test_query_2 = "problems with loan application"

print(f"Test Query with Filter: '{test_query_2}'")
print("Filter: Product = 'Personal loan'\n")

query_embedding_2 = embedder.generate_embedding(test_query_2)

results_filtered = vector_store.search(
    query_embedding=query_embedding_2,
    top_k=3,
    filter_dict={'Product': 'Personal loan'}
)

print(f"Found {len(results_filtered['documents'])} Personal loan chunks:\n")

for i, (doc, metadata, distance) in enumerate(
    zip(results_filtered['documents'], results_filtered['metadatas'], results_filtered['distances'])
):
    print(f"Result {i+1} (distance: {distance:.4f})")
    print(f"Text: {doc[:150]}...")
    print("-" * 80)
    print()

## Summary

### Task 2 Complete! ✓

We have successfully:

1. **Created Stratified Sample**: 12,000 complaints maintaining product distribution
2. **Chunked Text**: Split narratives into ~500 char chunks with 50 char overlap
3. **Generated Embeddings**: Used all-MiniLM-L6-v2 to create 384-dim vectors
4. **Built Vector Store**: Persisted embeddings in ChromaDB for semantic search
5. **Verified Functionality**: Tested similarity search with and without filters

### Key Metrics:
- Sample size: 12,000 documents
- Total chunks: ~[number] chunks
- Embedding dimension: 384
- Vector store location: `../vector_store/complaint_chunks`

### Design Decisions:
- **Chunk size (500)**: Balances context and granularity
- **Overlap (50)**: Preserves context at boundaries
- **Model (all-MiniLM-L6-v2)**: Fast, efficient, good quality
- **ChromaDB**: Easy to use, persistent, supports metadata filtering

The vector store is now ready for Task 3 (RAG Pipeline)!